# Week 4: Neural Network for the XOR Problem

This notebook implements a multilayer neural network from scratch using NumPy. The network learns XOR through forward propagation and backpropagation.

## Step 1 — Import Libraries and Create the Dataset

XOR returns `1` when its inputs differ and `0` when they are equal.

In [ ]:
# Import NumPy for arrays, matrix multiplication and calculations.
import numpy as np

# Import Matplotlib for visualising the training history.
import matplotlib.pyplot as plt

# Set the random seed so the starting weights are reproducible.
np.random.seed(42)

# Create the four possible XOR input combinations.
# Each row is one sample and each column is one input feature.
X = np.array([
    [0, 0],  # 0 XOR 0
    [0, 1],  # 0 XOR 1
    [1, 0],  # 1 XOR 0
    [1, 1]   # 1 XOR 1
], dtype=float)

# Store the correct output for each input combination.
y = np.array([
    [0],
    [1],
    [1],
    [0]
], dtype=float)

print("XOR inputs:")
print(X)
print("\nExpected outputs:")
print(y)
print("\nInput shape:", X.shape)
print("Output shape:", y.shape)

## Step 2 — Define the Neural Network

The network contains two input neurons, four hidden neurons and one output neuron. The class below keeps the complete implementation together so every method belongs to the same model.

In [ ]:
# Define the complete neural-network class.
class NeuralNetwork:

    # Create the weights and biases for all layers.
    def __init__(self, input_size, hidden_size, output_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size

        # Connect the two input neurons to the four hidden neurons.
        self.weights_input_hidden = np.random.randn(
            input_size, hidden_size
        )

        # Connect the four hidden neurons to the output neuron.
        self.weights_hidden_output = np.random.randn(
            hidden_size, output_size
        )

        # Start all bias values at zero.
        self.bias_hidden = np.zeros((1, hidden_size))
        self.bias_output = np.zeros((1, output_size))

        # Prepare lists for recording the training progress.
        self.loss_history = []
        self.accuracy_history = []

    # Convert a numerical value into a probability from 0 to 1.
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    # Calculate the slope of sigmoid during backpropagation.
    def sigmoid_derivative(self, sigmoid_output):
        return sigmoid_output * (1 - sigmoid_output)

    # Pass data forwards through the hidden and output layers.
    def feedforward(self, X):
        # Calculate and activate the hidden layer.
        self.hidden_activation = (
            np.dot(X, self.weights_input_hidden)
            + self.bias_hidden
        )
        self.hidden_output = self.sigmoid(self.hidden_activation)

        # Calculate and activate the output layer.
        self.output_activation = (
            np.dot(self.hidden_output, self.weights_hidden_output)
            + self.bias_output
        )
        self.predicted_output = self.sigmoid(self.output_activation)
        return self.predicted_output

    # Propagate the prediction error backwards and update parameters.
    def backward(self, X, y, learning_rate):
        # Calculate the output error and gradient.
        output_error = y - self.predicted_output
        output_delta = (
            output_error
            * self.sigmoid_derivative(self.predicted_output)
        )

        # Propagate the error into the hidden layer.
        hidden_error = np.dot(
            output_delta, self.weights_hidden_output.T
        )
        hidden_delta = (
            hidden_error
            * self.sigmoid_derivative(self.hidden_output)
        )

        # Update the hidden-to-output weights and output bias.
        self.weights_hidden_output += (
            np.dot(self.hidden_output.T, output_delta)
            * learning_rate
        )
        self.bias_output += (
            np.sum(output_delta, axis=0, keepdims=True)
            * learning_rate
        )

        # Update the input-to-hidden weights and hidden biases.
        self.weights_input_hidden += (
            np.dot(X.T, hidden_delta)
            * learning_rate
        )
        self.bias_hidden += (
            np.sum(hidden_delta, axis=0, keepdims=True)
            * learning_rate
        )

    # Train the network over multiple complete passes of the data.
    def train(self, X, y, epochs, learning_rate):
        # Clear any results left by earlier training.
        self.loss_history = []
        self.accuracy_history = []

        for epoch in range(epochs):
            # Produce predictions with the current parameters.
            output = self.feedforward(X)

            # Measure loss and binary classification accuracy.
            loss = np.mean(np.square(y - output))
            predictions = (output >= 0.5).astype(int)
            accuracy = np.mean(predictions == y)

            # Record the values for later visualisation.
            self.loss_history.append(loss)
            self.accuracy_history.append(accuracy)

            # Improve the weights and biases.
            self.backward(X, y, learning_rate)

            # Report selected epochs instead of printing all 10,000.
            if (
                epoch == 0
                or (epoch + 1) % 2000 == 0
                or epoch == epochs - 1
            ):
                print(
                    f"Epoch {epoch + 1:5d} | "
                    f"Loss: {loss:.6f} | "
                    f"Accuracy: {accuracy * 100:.1f}%"
                )

        return self


print("Complete NeuralNetwork class defined successfully.")

## Step 3 — Train the Neural Network

The network trains for 10,000 epochs with a learning rate of `0.1`. Each epoch performs forward propagation followed by backpropagation.

In [ ]:
# Reset the seed before creating the final model.
np.random.seed(42)

# Create a network with 2 inputs, 4 hidden neurons and 1 output.
nn = NeuralNetwork(
    input_size=2,
    hidden_size=4,
    output_size=1
)

# Train the network and create its loss and accuracy histories.
nn.train(
    X=X,
    y=y,
    epochs=10000,
    learning_rate=0.1
)

## Step 4 — Test the Trained Network

The sigmoid output gives a probability. A threshold of `0.5` converts each probability into class `0` or class `1`.

In [ ]:
# Produce final probabilities with the trained network.
final_probabilities = nn.feedforward(X)

# Convert the probabilities into binary predictions.
final_predictions = (final_probabilities >= 0.5).astype(int)

print("x1 | x2 | Probability | Prediction | Expected | Result")
print("-" * 65)

# Compare every prediction with the correct XOR answer.
for inputs, probability, prediction, expected in zip(
    X, final_probabilities, final_predictions, y
):
    x1, x2 = inputs
    probability_value = float(probability[0])
    predicted_class = int(prediction[0])
    expected_class = int(expected[0])
    result = (
        "Correct"
        if predicted_class == expected_class
        else "Incorrect"
    )

    print(
        f"{int(x1):>2} | {int(x2):>2} | "
        f"{probability_value:>11.4f} | "
        f"{predicted_class:>10} | "
        f"{expected_class:>8} | {result}"
    )

final_accuracy = np.mean(final_predictions == y)
print("-" * 65)
print(f"Final accuracy: {final_accuracy * 100:.1f}%")

## Step 5 — Visualise the Training Progress

Loss should decrease towards zero while accuracy rises to 100%.

In [ ]:
# Create epoch numbers beginning at 1.
epoch_numbers = range(1, len(nn.loss_history) + 1)

# Create two graphs side by side.
figure, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot mean squared error during training.
axes[0].plot(
    epoch_numbers,
    nn.loss_history,
    color="steelblue",
    linewidth=2
)
axes[0].set_title("XOR Neural Network Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Mean Squared Error")
axes[0].grid(True, linestyle="--", alpha=0.5)

# Convert decimal accuracy values into percentages.
accuracy_percentages = [
    accuracy * 100 for accuracy in nn.accuracy_history
]

# Plot classification accuracy during training.
axes[1].plot(
    epoch_numbers,
    accuracy_percentages,
    color="seagreen",
    linewidth=2
)
axes[1].set_title("XOR Neural Network Training Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_ylim(0, 105)
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Conclusion

The multilayer network learned the non-linear XOR relationship. Four hidden neurons transformed the inputs, and backpropagation adjusted both sets of weights and biases until all four samples were classified correctly. A single perceptron cannot solve XOR because one straight decision boundary cannot separate its classes.